# Retail Inventory Analysis

**Scenario**: You're working at a retail chain analyzing inventory turnover. The store manager needs to understand stock movement patterns to optimize ordering.

**Data Structure: DataFrame**

- product_id
- category
- stock_level
- last_restock_date
- sales_last_30_days
- supplier_lead_time
- unit_cost

**Task: Generate code to:**

- Calculate inventory turnover rates.

1. **How do we calculate "Inventory Turnover Rate" with this data?**

In accounting, the **Inventory Turnover Ratio** is typically defined as:

![Inventory Turnover Ratio](inventory_turnover_eq.png)

However, looking at our DataFrame schema:

* We only have `stock_level` (which is current/static, not historical average).
* We have `sales_last_30_days` (which is likely a unit count, though it could be revenue—let's assume units for now) and `unit_cost`.

**The Debate**:

* **Option A (Value-based)**: We estimate monthly COGS as `sales_last_30_days * unit_cost` and divide it by the current inventory value (`stock_level * unit_cost`). This simplifies to `sales_last_30_days / stock_level`.
* **Option B (Unit-based)**: We simply use the ratio of unit sales to current stock: `sales_last_30_days / stock_level`.
* **The Catch**: What if `stock_level` is 0? We will get a division-by-zero error (or `inf` in pandas).

*How do you think we should define turnover here? And how should we handle cases where current stock is zero?*

- Identify slow-moving items.

2. **What actually makes an item "slow-moving"?**

Identifying slow-moving items isn't always as simple as looking for low sales.

**The Debate**:

* If a product has 0 sales in the last 30 days, is it slow-moving?
* What if `last_restock_date` was only 2 days ago? Maybe it's a brand-new item that just arrived on the shelves, so it hasn't had time to sell yet.
* Conversely, if an item has a high `stock_level`, 0 sales, and the `last_restock_date` was 6 months ago, that is a clear red flag.

*How should we combine `sales_last_30_days`, `stock_level`, and `last_restock_date` to define "slow-moving"? Do we set a threshold of days-since-last-restock, or do we define it purely by how many months of inventory we currently hold?*

- Predict potential stockouts.

3. **How do we predict "potential stockouts" realistically?**

A standard way to predict a stockout is to calculate **Days of Inventory Left (DoI)** and compare it to the **Supplier Lead Time**.

![Days of Inventory Left(Dol)](days_of_inventory_left_calc.png)

​If Days of Inventory ≤ supplier_lead_time, we risk running out of stock before a new order arrives.

**The Debate:**

* What if `sales_last_30_days` is 0? The daily sales rate is 0, which makes Days of Inventory infinite. But if stock is also low, is it really safe from a stockout, or is it just a dead product?
* Lead time is a crucial buffer. Should we build in a "safety stock" buffer (e.g., triggering a warning if stock will run out within `lead_time + 5 days`), or stick strictly to the lead time?

*What rules should we establish for our stockout prediction logic?*

- Create visualizations.

4. **What is the most effective way to visualize this for a manager?**

The prompt asks for visualizations. A store manager doesn't want to look at a chaotic chart containing hundreds of individual product_id bars.

**The Debate:**

* Should we aggregate the data by `category` to show which categories have the worst turnover or highest stockout risk?
* Should we use a scatter plot mapping `stock_level` against `sales_last_30_days` to visually isolate the "problem quadrants" (e.g., high stock / low sales vs. low stock / high sales)?

*What kind of visual presentation do you think would be most intuitive and actionable for a retail manager?*

----

**My Response to the above questions**

How do you think we should define turnover here? And how should we handle cases where current stock is zero? we should use the Inventory Turnover Ratio and ensure that we don't to having a zero stock, we set a rule that doesn't allow stock item quantity not go beyond a certain threshold say 5

How should we combine sales_last_30_days, stock_level, and last_restock_date to define "slow-moving"? Do we set a threshold of days-since-last-restock, or do we define it purely by how many months of inventory we currently hold? From what I have encountered myself selling for a friend in their shop, a slow moving stock, is kind of one that takes too long in the store or like it's demand is low, people scramble for it on launch but then eventually stop ordering or purchasing it; 

If a product has 0 sales in the last 30 days, is it slow-moving? yes it is a slow moving; 

What if last_restock_date was only 2 days ago? Maybe it's a brand-new item that just arrived on the shelves, so it hasn't had time to sell yet. I hear your argument here, so let's not use this approach, if it is new we can give a certain timespan like 2 weeks if with 0 sales, after which we can categorize it as slow moving

For the Predict potential stockouts there you'll have to break it down for me in laymans language

What kind of visual presentation do you think would be most intuitive and actionable for a retail manager? For this, give me options with reasons why they would fit the scenario and then I can choose from that

----

**Continuation of specs architecturing**

### 1. Handling Turnover and the Zero Stock Threshold

Your real-world observation about setting a minimum stock threshold is highly practical. If stock levels drop too low, it represents a breakdown in the supply chain, not just a mathematical issue. 

Based on your input, here is how we can translate this into code logic:
* **The "Safety Stock" Threshold:** We will define a safety threshold of **5 units**. 
* **Flagging Low Stock:** Any product with a `stock_level` below 5 is flagged as an urgent restock priority.
* **Calculating Turnover safely:** To calculate the **Inventory Turnover Ratio** ($\text{Sales} / \text{Stock}$) without triggering division-by-zero errors when stock is 0, we can write a rule in Python:
  * If the stock is greater than 0, calculate: `sales_last_30_days / stock_level`.
  * If the stock is 0, we can output `0` (or `NaN`) for the turnover and flag it as "Out of Stock" so it doesn't break our code.

---

### 2. The Slow-Moving Rule (with a 14-Day Grace Period)

Your point about giving new items a chance to perform is an excellent adjustment. If an item was restocked just a few days ago, it hasn't had a fair window of time to generate sales. 

We will define **"Slow-Moving"** using a two-part rule:
1. **Low Demand:** The product has had **0 sales** in the last 30 days.
2. **The 14-Day Grace Period:** The product was restocked **at least 14 days ago** (meaning `current_date - last_restock_date >= 14 days`). 

*Note on date calculation:* Since a standard DataFrame doesn't know today's exact date, we will establish a reference "current date" in the code (usually the most recent date in the dataset) to calculate how many days have passed since the `last_restock_date` [1].

---

### 3. Predicting Potential Stockouts (In Layman's Terms)

Imagine you run a grocery store and sell bread:

1. **Your Stock:** You currently have **10 loaves of bread** on your shelf.
2. **Your Sales Speed:** On average, customers buy **2 loaves of bread every day**.
3. **Days Left:** At this speed, your bread will last for **5 days** (10 loaves divided by 2 loaves per day).
4. **Supplier Lead Time:** You call the bakery to order more bread. They tell you it takes **7 days** for the delivery truck to arrive with a new batch.
5. **The Stockout Risk:** Your bread will run out in 5 days, but the truck won't arrive for 7 days. This means your shelf will be completely empty for 2 days. 

In our code, we will flag a **potential stockout** whenever the **Days of Stock Left** is less than the **Supplier Lead Time**.

---

### 4. Visualization Options for the Store Manager

To help the store manager make quick, smart decisions, here are three intuitive options. Please review them and let me know which one(s) you would prefer to generate:

#### **Option A: The "Actionable Reorder" Horizontal Bar Chart**
* **What it shows:** A list of the top 10–15 products most at risk of running out of stock. For each product, it shows two bars side-by-side: how many days of stock are left vs. how many days the supplier takes to deliver.
* **Why it fits:** It is highly practical. A manager can look at this chart every Monday morning and immediately see exactly which products must be ordered today to prevent empty shelves.

#### **Option B: The "Four-Quadrant" Inventory Scatter Plot**
* **What it shows:** Every product plotted as a dot. The vertical axis is the stock level; the horizontal axis is sales speed.
  * **Top-Left (Low stock, high sales):** The "Hot Sellers" that need immediate restock.
  * **Bottom-Right (High stock, low sales):** The "Dead Weight" (slow-moving items taking up shelf space).
* **Why it fits:** It gives the manager a complete visual map of the entire store's inventory health in a single glance.

#### **Option C: Category Health Bar Chart**
* **What it shows:** A high-level chart grouping products by category (e.g., Electronics, Grocery, Apparel) showing the average turnover rate of each category.
* **Why it fits:** It is great for big-picture planning. It tells the manager which departments are performing well and which ones are stagnant and might need promotions or discounts to clear space.

---

In [ ]:
# Your Prompt:
"""
Write Python code to analyze a retail inventory DataFrame.
1. Calculate the inventory turnover rate (sales divided by stock).
2. Identify slow-moving items (items with 0 sales).
3. Predict potential stockouts (when days of stock left is less than supplier lead time).
4. Generate a scatter plot of stock vs. sales and a bar chart of turnover rate by category.
"""

# Generated Code:


df['turnover_rate'] = df['sales_last_30_days'] / df['stock_level']  # Risk: Division by zero if stock is 0

df['slow_moving'] = df['sales_last_30_days'] == 0  # Risk: Flags brand new arrivals as slow-moving

# Risk: Division by zero if sales are 0
df['daily_sales_rate'] = df['sales_last_30_days'] / 30.0
df['days_of_stock_left'] = df['stock_level'] / df['daily_sales_rate']
df['potential_stockout'] = df['days_of_stock_left'] < df['supplier_lead_time']

# Separate plots
plt.figure()
sns.scatterplot(data=df, x='sales_last_30_days', y='stock_level')
plt.title("Stock vs Sales")
plt.show()

plt.figure()
sns.barplot(data=df, x='category', y='turnover_rate')
plt.title("Turnover by Category")
plt.show()


# Follow-up Prompts:
"""
1. "How do we handle division by zero in the turnover calculation when stock is 0? Let's treat stock levels below 5 as an immediate restock trigger, and safely compute turnover."
2. "If a product is restocked less than 14 days ago, it shouldn't be called slow-moving even if sales are 0. Please update the slow-moving logic to include a 14-day grace period using `last_restock_date`."
3. "Combine the visualizations side-by-side in a 1x2 subplot: on the left, a scatter plot of stock level vs sales; on the right, a bar chart of average turnover by category."
"""

# Final Solution:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------------------
# 1. Create Dummy Retail Data (Designed to test edge cases)
# -------------------------------------------------------------------------
# Evaluation Date: Set to 2026-06-30 to run historical calculations
evaluation_date = pd.to_datetime('2026-06-30')

data = {
    'product_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008'],
    'category': ['Grocery', 'Electronics', 'Grocery', 'Apparel', 'Electronics', 'Apparel', 'Grocery', 'Electronics'],
    'stock_level': [50, 0, 3, 100, 4, 15, 80, 12],
    'last_restock_date': pd.to_datetime([
        '2026-06-10',  # 20 days ago (Normal)
        '2026-05-15',  # 46 days ago (Out of stock)
        '2026-06-28',  # 2 days ago (New arrival - grace period candidate)
        '2026-04-01',  # 90 days ago (Old stock)
        '2026-06-29',  # 1 day ago (New arrival - grace period candidate)
        '2026-06-15',  # 15 days ago (Normal)
        '2026-06-20',  # 10 days ago (High performing Grocery)
        '2026-06-01'   # 29 days ago (Normal)
    ]),
    'sales_last_30_days': [120, 0, 0, 5, 0, 30, 150, 40],
    'supplier_lead_time': [5, 10, 3, 15, 7, 5, 4, 10],
    'unit_cost': [2.50, 150.00, 1.20, 25.00, 80.00, 18.00, 3.00, 95.00]
}

df = pd.DataFrame(data)

# -------------------------------------------------------------------------
# 2. Robust Analysis Calculations
# -------------------------------------------------------------------------

# A. Safety Stock & Restock Alert (Threshold = 5)
df['low_stock_warning'] = df['stock_level'] < 5

# B. Inventory Turnover Rate (Handles stock_level = 0 safely)
df['turnover_rate'] = np.where(
    df['stock_level'] > 0, 
    df['sales_last_30_days'] / df['stock_level'], 
    0.0
)

# C. Slow-Moving Items (With 14-day grace period for newly restocked items)
df['days_since_restock'] = (evaluation_date - df['last_restock_date']).dt.days
df['slow_moving'] = (df['sales_last_30_days'] == 0) & (df['days_since_restock'] >= 14)

# D. Stockout Predictions (Handles daily sales rate = 0 safely)
df['daily_sales_rate'] = df['sales_last_30_days'] / 30.0

df['days_of_stock_left'] = np.where(
    df['daily_sales_rate'] > 0, 
    df['stock_level'] / df['daily_sales_rate'], 
    np.inf  # If there are no sales, stock mathematically lasts indefinitely
)

df['potential_stockout'] = df['days_of_stock_left'] < df['supplier_lead_time']

# -------------------------------------------------------------------------
# 3. Side-by-Side Visualizations (Options B & C Combined)
# -------------------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left Plot: Option B (Four-Quadrant Scatter Plot for Stock Level vs. Sales) ---
sns.scatterplot(
    data=df, 
    x='sales_last_30_days', 
    y='stock_level', 
    hue='potential_stockout', 
    palette={True: '#e74c3c', False: '#3498db'},
    s=150, 
    ax=axes[0]
)

# Reference Threshold Lines for Quadrants
axes[0].axhline(y=5, color='gray', linestyle='--', alpha=0.7, label='Safety Stock Threshold (5)')
axes[0].axvline(x=15, color='gray', linestyle=':', alpha=0.7, label='Sales Threshold (15)')

# Labeling quadrants for ease of reading
axes[0].text(30, 55, 'High Stock / High Sales\n(Healthy Status)', fontsize=10, color='green', bbox=dict(facecolor='white', alpha=0.6))
axes[0].text(30, 8, 'Low Stock / High Sales\n(Urgent Stockout Risk!)', fontsize=10, color='darkred', bbox=dict(facecolor='white', alpha=0.6))
axes[0].text(1, 80, 'High Stock / Low Sales\n(Slow Moving / Overstocked)', fontsize=10, color='orange', bbox=dict(facecolor='white', alpha=0.6))

axes[0].set_title('Stock Levels vs. Sales Performance (Option B)', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Units Sold (Last 30 Days)', fontsize=11)
axes[0].set_ylabel('Current Stock Level', fontsize=11)
axes[0].legend(title='Stockout Warning Status', loc='upper right')

# --- Right Plot: Option C (Category Health Bar Chart) ---
category_summary = df.groupby('category')['turnover_rate'].mean().reset_index()

sns.barplot(
    data=category_summary, 
    x='category', 
    y='turnover_rate', 
    hue='category',
    palette='viridis', 
    ax=axes[1],
    legend=False
)

axes[1].set_title('Average Turnover Rate by Category (Option C)', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Product Category', fontsize=11)
axes[1].set_ylabel('Average Turnover (Sales / Stock Ratio)', fontsize=11)

# Annotate values on top of the bars
for p in axes[1].patches:
    height = p.get_height()
    axes[1].annotate(
        f'{height:.2f}x',
        (p.get_x() + p.get_width() / 2., height),
        ha='center', va='center',
        xytext=(0, 9),
        textcoords='offset points',
        fontsize=10,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

# Display the processed, easy-to-read audit summary
print("--- PROCESSED INVENTORY ANALYSIS AUDIT ---")
print(df[['product_id', 'category', 'stock_level', 'sales_last_30_days', 'turnover_rate', 'slow_moving', 'potential_stockout']])

**Discussion Questions to Answer:**
---

### 1. How did different prompts handle date calculations?
* **The Initial Prompt:** The naive prompt completely ignored temporal context. It categorized any product with zero sales as "slow-moving" without considering when the item was placed on the shelf.
* **The Refined Prompt:** The follow-up prompt introduced a business-driven temporal rule. It calculated the elapsed time since the last shipment arrived using:
  $$\text{Days Since Restock} = \text{Evaluation Date} - \text{Last Restock Date}$$
  By setting a **14-day grace period**, the code was updated to only flag an item as slow-moving if it had zero sales *and* had been sitting on the shelves for at least two weeks, preventing brand-new products from being flagged prematurely [1].

---

### 2. What visualization approaches were suggested?
* **The Initial Suggestion:** The basic approach generated separate, standalone plots that a non-technical manager might find difficult to parse cohesively (such as individual scatter plots and unlabelled category bar charts).
* **The Refined Suggestion:** To make the data highly intuitive and actionable for a manager, we combined two complimentary approaches side-by-side in a single $1 \times 2$ grid:
  * **The "Four-Quadrant" Scatter Plot (Left):** Plotted stock levels against sales, dividing the chart with threshold lines (such as a safety stock line at 5 units). This visually grouped products into distinct management categories (e.g., *Healthy Sellers* vs. *Urgent Stockout Risks* vs. *Slow-Moving Overstock*).
  * **The Category Health Bar Chart (Right):** Summarized performance at a department level by plotting the average turnover rate for each category, complete with bold labels on top of each bar for immediate reading.

---

### 3. How was error handling implemented?
* **The Initial Approach:** The naive code relied on direct mathematical division (e.g., `sales / stock`). If a product was out of stock (`stock_level = 0`) or had zero sales (`daily_sales = 0`), the code would either crash or produce messy `inf` and `NaN` values.
* **The Refined Approach:** Robust error handling was implemented using conditional vector operations (`numpy.where`) to catch division-by-zero errors before they happened:
  * **Zero Stock Handling:** The code checked if the stock level was greater than zero. If the stock level was `0`, it assigned a turnover rate of `0.0` rather than attempting a invalid division.
  * **Zero Sales Handling:** When estimating days of stock left, the code checked if the daily sales rate was greater than zero. If a product had zero sales, it safely assigned an infinite shelf-life (`np.inf`) to prevent a division error, meaning it would not trigger a false "potential stockout" alert.